# Subsystem memory: Bacon–Shor and SHYPS

**[DEMO]** — this notebook uses the packaged `MemoryExperiment` protocol and code classes.

The patch declares the stabilizer centre and gauge generators once. The default extraction measures X gauges and then Z gauges, and LightStim derives detectors from their measurement history. No manual changes to the active checks are needed between those groups.

We build two-round X/Z memories for Bacon–Shor `[[9,1,4,3]]` (the third entry counts gauge qubits) and SHYPS `[[49,9,4]]` (data, protected logicals, distance). SHYPS uses 147 physical qubits here, including its 98 gauge-readout ancillas. These are integration checks; code distance is not a claim about the fault distance of this extraction schedule.

In [1]:
import sys
from pathlib import Path

ROOT = next(
    path
    for path in (Path.cwd(), *Path.cwd().parents)
    if (path / "lightstim").is_dir() and (path / "pyproject.toml").exists()
)
if str(ROOT) not in sys.path:
    sys.path.insert(0, str(ROOT))

from lightstim.noise.config import NoiseConfig
from lightstim.protocols.memory import MemoryExperiment
from lightstim.qec_code.bacon_shor import BaconShorCode
from lightstim.qec_code.shyps import SHYPSCode

## Ideal memory checks

Each returned observable belongs to a protected logical qubit. Gauge-state directions are excluded from that count. The small ideal samples below must contain no detector or observable flips, and the detector error model must be constructible.

In [2]:
def check_ideal_memory(circuit, expected_logicals):
    detectors, observables = circuit.compile_detector_sampler(seed=71).sample(
        128, separate_observables=True
    )
    assert circuit.num_observables == expected_logicals
    assert observables.shape == (128, expected_logicals)
    assert not detectors.any()
    assert not observables.any()
    assert circuit.detector_error_model().num_observables == expected_logicals
    return {
        "physical_qubits": circuit.num_qubits,
        "detectors": circuit.num_detectors,
        "protected_observables": circuit.num_observables,
        "ideal_detector_flips": int(detectors.sum()),
        "ideal_observable_flips": int(observables.sum()),
    }

bacon_z = MemoryExperiment(
    qec_patch=BaconShorCode(distance=3), rounds=2, basis="Z"
).build()
bacon_x = MemoryExperiment(
    qec_patch=BaconShorCode(distance=3), rounds=2, basis="X"
).build()
print("Bacon–Shor Z:", check_ideal_memory(bacon_z, 1))
print("Bacon–Shor X:", check_ideal_memory(bacon_x, 1))

# Optional small circuit visualization; keep its output cleared when saving.
# bacon_z.diagram("detslice-with-ops-svg")

Bacon–Shor Z: {'physical_qubits': 21, 'detectors': 12, 'protected_observables': 1, 'ideal_detector_flips': 0, 'ideal_observable_flips': 0}
Bacon–Shor X: {'physical_qubits': 21, 'detectors': 12, 'protected_observables': 1, 'ideal_detector_flips': 0, 'ideal_observable_flips': 0}


In [3]:
shyps_z = MemoryExperiment(
    qec_patch=SHYPSCode(r=3), rounds=2, basis="Z"
).build()
shyps_x = MemoryExperiment(
    qec_patch=SHYPSCode(r=3), rounds=2, basis="X"
).build()
print("SHYPS Z:", check_ideal_memory(shyps_z, 9))
print("SHYPS X:", check_ideal_memory(shyps_x, 9))

SHYPS Z: {'physical_qubits': 147, 'detectors': 148, 'protected_observables': 9, 'ideal_detector_flips': 0, 'ideal_observable_flips': 0}
SHYPS X: {'physical_qubits': 147, 'detectors': 148, 'protected_observables': 9, 'ideal_detector_flips': 0, 'ideal_observable_flips': 0}


## Circuit noise and detector error models

The same public protocol accepts a `NoiseConfig`. Here we add two-qubit and readout noise, then construct each DEM without requesting graphlike decomposition. Sampling or constructing a DEM is not decoding; these counts do not establish a logical error rate, threshold or single-shot performance.

In [4]:
noise = NoiseConfig(p_2q=0.001, p_meas=0.001)

bacon_noisy = MemoryExperiment(
    qec_patch=BaconShorCode(distance=3), rounds=2, basis="Z",
    noise_params=noise,
).build()
shyps_noisy = MemoryExperiment(
    qec_patch=SHYPSCode(r=3), rounds=2, basis="Z",
    noise_params=noise,
).build()

bacon_dem = bacon_noisy.detector_error_model()
shyps_dem = shyps_noisy.detector_error_model()
assert bacon_dem.num_errors > 0 and bacon_dem.num_observables == 1
assert shyps_dem.num_errors > 0 and shyps_dem.num_observables == 9
print("Bacon–Shor DEM:", {"error_terms": bacon_dem.num_errors, "observables": bacon_dem.num_observables})
print("SHYPS DEM:", {"error_terms": shyps_dem.num_errors, "observables": shyps_dem.num_observables})

Bacon–Shor DEM: {'error_terms': 58, 'observables': 1}
SHYPS DEM: {'error_terms': 2009, 'observables': 9}


## Construction references

- [Aliferis and Cross, *Subsystem fault tolerance with the Bacon–Shor code*](https://arxiv.org/html/quant-ph/0610063v3): low-weight gauge measurements and protected/gauge degrees of freedom.
- [Malcolm et al., *Computing Efficiently in QLDPC Codes*, §§VIII.4–VIII.5](https://arxiv.org/html/2502.07150v2): the SHYPS matrices, parameters and paired bare logical representatives.
- [SHYPS implementation notes](../../lightstim/qec_code/shyps/README.md): supported sizes, indexing and ancilla allocation.

The SHYPS default is a generic finite gauge-extraction schedule. It does not reproduce the paper's logical Clifford compiler, decoder or optimized performance results.